In [12]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "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"
)


completion = client.chat.completions.create(
  model="nvidia/nemotron-3-ultra-550b-a55b",
  messages=[{"role":"user","content":"Write a limerick about the wonders of GPU computing."}],
  temperature=1,
  top_p=0.95,
  max_tokens=16384,
  extra_body={"chat_template_kwargs":{"enable_thinking":True}},
  stream=True
)

for chunk in completion:
  if not chunk.choices:
    continue
  reasoning = getattr(chunk.choices[0].delta, "reasoning_content", None)
  if reasoning:
    print(reasoning, end="")
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

The user wants a limerick about GPU computing.
A limerick has an AABBA rhyme scheme and a specific rhythm (anapestic meter).A CPU works in a line,  
Completing each task just in time.  
But a GPU’s might  
Runs thousands in light,  
Parallel power—sublime!

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


base_url = "https://integrate.api.nvidia.com/v1" 
api_key = "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"  
model = "nvidia/nemotron-3-ultra-550b-a55b" 


llm = ChatOpenAI(
    model=model,
    base_url=base_url,
    api_key=api_key,
    
)

# ۳. ارسال یک پیام تست
try:
    response = llm.invoke([HumanMessage(content="فقط بنویس: اتصال برقرار شد")])
    print("✅ اتصال موفق بود. پاسخ مدل:")
    print(response.content)
except Exception as e:
    print(f"❌ خطا در اتصال: {e}")

✅ اتصال موفق بود. پاسخ مدل:
اتصال برقرار شد


In [15]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root added to sys.path:", project_root)

Project root added to sys.path: e:\new_world\projects\rag-engine


In [16]:
import os
from src import config
from dotenv import load_dotenv

load_dotenv()

print("API_KEY configured:", bool(os.getenv("API_KEY")))
print("TAVILY_API_KEY configured:", bool(os.getenv("TAVILY_API_KEY")))
print("PINECONE_API_KEY configured:", bool(os.getenv("PINECONE_API_KEY")))

API_KEY configured: True
TAVILY_API_KEY configured: True
PINECONE_API_KEY configured: True


In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage


base_url = "https://integrate.api.nvidia.com/v1" 
api_key = "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"  
model = "nvidia/nemotron-3-ultra-550b-a55b" 


llm = ChatOpenAI(
    model=config.MODEL,
    base_url=config.BASE_URL,
    api_key=config.API_KEY,   
    temperature=0,
)

# ۳. ارسال یک پیام تست
try:
    response = llm.invoke([HumanMessage(content="فقط بنویس: اتصال برقرار شد")])
    print("✅ اتصال موفق بود. پاسخ مدل:")
    print(response.content)
except Exception as e:
    print(f"❌ خطا در اتصال: {e}")

✅ اتصال موفق بود. پاسخ مدل:
اتصال برقرار شد


In [23]:
import time
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from src import config

PROMPT = "فقط بنویس: اتصال برقرار شد"
N_RUNS = 3  # چند بار تکرار شود تا نویز شبکه میانگین‌گیری شود

def benchmark(name, base_url, api_key, model, temperature=0):
    """زمان‌بندی یک فراخوانی غیراستریمی."""
    llm = ChatOpenAI(
        model=model,
        base_url=base_url,
        api_key=api_key,
        temperature=temperature,
        max_retries=0,  # برای اینکه خطاها زمان‌بندی را خراب نکنند
    )
    times = []
    for i in range(N_RUNS):
        t0 = time.perf_counter()
        try:
            resp = llm.invoke([HumanMessage(content=PROMPT)])
            elapsed = time.perf_counter() - t0
            times.append(elapsed)
            print(f"  [{name}] run {i+1}: {elapsed:.3f}s → {resp.content!r}")
        except Exception as e:
            elapsed = time.perf_counter() - t0
            print(f"  [{name}] run {i+1}: FAILED after {elapsed:.3f}s → {e}")
        time.sleep(0.5)  # فاصله بین درخواست‌ها

    if times:
        print(f"  [{name}] میانگین: {sum(times)/len(times):.3f}s | "
              f"حداقل: {min(times):.3f}s | حداکثر: {max(times):.3f}s\n")
    return times


print("=" * 60)
print("۱) API انویدیا (مستقیم)")
print("=" * 60)
nvidia_times = benchmark(
    name="NVIDIA",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="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",
    model="nvidia/nemotron-3-ultra-550b-a55b",
)

print("=" * 60)
print("۲) API داخل کانفیگ (config)")
print("=" * 60)
config_times = benchmark(
    name="config",
    base_url=config.BASE_URL,
    api_key=config.API_KEY,
    model=config.MODEL,
)

# مقایسه نهایی
if nvidia_times and config_times:
    nv_avg = sum(nvidia_times) / len(nvidia_times)
    cf_avg = sum(config_times) / len(config_times)
    print("=" * 60)
    print(f"نتیجه: NVIDIA = {nv_avg:.3f}s | config = {cf_avg:.3f}s | "
          f"اختلاف = {cf_avg - nv_avg:+.3f}s")

۱) API انویدیا (مستقیم)
  [NVIDIA] run 1: 1.830s → 'اتصال برقرار شد'
  [NVIDIA] run 2: 8.402s → 'اتصال برقرار شد'
  [NVIDIA] run 3: 0.866s → 'اتصال برقرار شد'
  [NVIDIA] میانگین: 3.699s | حداقل: 0.866s | حداکثر: 8.402s

۲) API داخل کانفیگ (config)
  [config] run 1: 3.355s → 'اتصال برقرار شد'
  [config] run 2: 6.600s → 'اتصال برقرار شد'
  [config] run 3: 2.964s → 'اتصال برقرار شد'
  [config] میانگین: 4.307s | حداقل: 2.964s | حداکثر: 6.600s

نتیجه: NVIDIA = 3.699s | config = 4.307s | اختلاف = +0.607s
